In [1]:
import json
import time
import getpass
import urllib.request
import urllib.parse
from pathlib import Path
from datetime import datetime, timedelta, timezone

api_key = getpass.getpass("Paste your TwitterAPI.io key: ").strip()
output = Path("twitter_pilot")
output.mkdir(exist_ok=True)

# New York was UTC−4 during this September window.
ny = timezone(timedelta(hours=-4))
results = []

try:
    for i, day in enumerate(["2026-09-08", "2026-09-19", "2026-09-30"]):
        start = datetime.fromisoformat(day).replace(tzinfo=ny)
        end = start + timedelta(days=1)

        query = (
            '("artificial intelligence" OR "generative AI") lang:en '
            f"since_time:{int(start.timestamp())} "
            f"until_time:{int(end.timestamp())}"
        )

        params = urllib.parse.urlencode({
            "query": query,
            "queryType": "Latest",
            "cursor": ""
        })

        request = urllib.request.Request(
            "https://api.twitterapi.io/twitter/tweet/advanced_search?" + params,
            headers={"X-API-Key": api_key}
        )

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                data = json.load(response)
        except urllib.error.HTTPError as error:
            print(f"{day}: HTTP {error.code}. Stopping—send me this code.")
            break

        record = {
            "requested_date_new_york": day,
            "query": query,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "response": data
        }
        results.append(record)
        (output / f"{day}.json").write_text(
            json.dumps(record, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        tweets = data.get("tweets", [])
        print(f"{day}: {len(tweets)} posts returned")

        if tweets:
            first = tweets[0]
            print("First timestamp:", first.get("createdAt"))
            print("Example:", first.get("text", "")[:180], "\n")
        else:
            print("Response status:", data.get("status"),
                  "| Message:", data.get("message"))

        if i < 2:
            time.sleep(21)

finally:
    api_key = None

Path("twitter_pilot_results.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Saved: twitter_pilot_results.json")

2026-09-08: 20 posts returned
First timestamp: Wed Sep 09 03:59:42 +0000 2026
Example: @Jeffreyhbaird @Hitchslap1 Intelligence is what we should be focussed mostly on, especially as we try to build artificial intelligence. Word olympics don't change reality. 

2026-09-19: 20 posts returned
First timestamp: Sun Sep 20 03:59:19 +0000 2026
Example: I'm LIVE🔴 I'm so exhausted as a small creator trying to succeed in a space where the use of generative AI in games is the norm..:( Let's de-stress and play some dinos..?

( Stream  

2026-09-30: 20 posts returned
First timestamp: Thu Oct 01 03:59:31 +0000 2026
Example: @GavinNewsom I don't think people realize the extreme irony of one of the dumbest people on earth trying to rename artificial intelligence to super intelligence. The entire right s 

Saved: twitter_pilot_results.json


In [ ]:
import json
import time
import random
import getpass
import urllib.request
import urllib.parse
import urllib.error
from pathlib import Path
from datetime import datetime, timedelta, timezone

# ---------- SETTINGS ----------
folder = Path("twitter_collection_v1")
folder.mkdir(exist_ok=True)

ny = timezone(timedelta(hours=-4))  # New York in September 2026
rng = random.Random(20260908)

base_query = '("artificial intelligence" OR "generative AI") lang:en'
endpoint = "https://api.twitterapi.io/twitter/tweet/advanced_search"

# Two requests per day:
# Key 1: 22 requests, about 440 posts / 6,600 credits.
# Key 2: 24 requests, about 480 posts / 7,200 credits.
# Estimates assume the 20-post response size observed in our pilot.
# No pagination or automatic retries in this first collection.

keys = [
    getpass.getpass("KEY 1 — paste key with 9.1k credits: ").strip(),
    getpass.getpass("KEY 2 — paste key with 10k credits: ").strip()
]

if not all(keys):
    raise ValueError("Both keys are required.")

if keys[0] == keys[1]:
    raise ValueError("You entered the same key twice. Use the two different keys.")

# Build the same sampling schedule every time.
jobs = []

for day_number in range(8, 31):
    midnight = datetime(2026, 9, day_number, tzinfo=ny)
    key_index = 0 if day_number <= 18 else 1

    for half in range(2):
        # Random one-hour window wholly inside each 12-hour half-day.
        offset_minutes = half * 720 + rng.randint(0, 660)
        start = midnight + timedelta(minutes=offset_minutes)
        end = start + timedelta(hours=1)

        query = (
            f"{base_query} "
            f"since_time:{int(start.timestamp())} "
            f"until_time:{int(end.timestamp())}"
        )

        jobs.append({
            "date": midnight.date().isoformat(),
            "window": half + 1,
            "key_index": key_index,
            "start": start.isoformat(),
            "end": end.isoformat(),
            "query": query
        })

last_request_time = None

try:
    for number, job in enumerate(jobs, start=1):
        filename = folder / f"{job['date']}_window{job['window']}.json"

        if filename.exists():
            saved = json.loads(filename.read_text(encoding="utf-8"))
            if saved.get("query") != job["query"]:
                raise RuntimeError("Saved query differs. Stop and ask me to check.")
            print(f"[{number}/46] Already saved: {filename.name}")
            continue

        # Conservative spacing between requests.
        if last_request_time is not None:
            remaining = 21 - (time.monotonic() - last_request_time)
            if remaining > 0:
                time.sleep(remaining)

        params = urllib.parse.urlencode({
            "query": job["query"],
            "queryType": "Latest",
            "cursor": ""
        })

        request = urllib.request.Request(
            endpoint + "?" + params,
            headers={"X-API-Key": keys[job["key_index"]]}
        )

        last_request_time = time.monotonic()

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                data = json.load(response)

        except urllib.error.HTTPError as error:
            print(
                f"STOPPED: HTTP {error.code} on {job['date']}. "
                "Completed downloads are saved. Send me this error code."
            )
            break

        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError):
            print(
                "STOPPED: connection, timeout, or response-format problem. "
                "Completed downloads are saved. Send me this message."
            )
            break

        if not isinstance(data, dict) or not isinstance(data.get("tweets"), list):
            print("STOPPED: unexpected API response. No further requests made.")
            break

        if str(data.get("status", "")).lower() in {"error", "failed", "fail"}:
            print("STOPPED: API reported an error. No further requests made.")
            break

        record = {
            "requested_date_new_york": job["date"],
            "window_number": job["window"],
            "window_start_new_york": job["start"],
            "window_end_new_york": job["end"],
            "credential_label": f"key_{job['key_index'] + 1}",
            "query": job["query"],
            "sampling": "First Latest page from a seeded one-hour window",
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "response": data
        }

        # Save immediately; no API keys or request headers are saved.
        temporary = filename.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(record, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        temporary.replace(filename)

        count = len(data["tweets"])
        print(
            f"[{number}/46] {job['date']} | "
            f"window {job['window']} | key {job['key_index'] + 1} | "
            f"{count} posts"
        )

        # Stop if response sizes exceed what we budgeted from the pilot.
        if count > 20:
            print(
                "STOPPED: API returned more than 20 posts in one request. "
                "Send me the results so we can revise the credit estimate."
            )
            break

except KeyboardInterrupt:
    print("\nPaused. Completed downloads are saved.")

finally:
    keys.clear()
    request = None

    records = [
        json.loads(path.read_text(encoding="utf-8"))
        for path in sorted(folder.glob("*_window*.json"))
    ]

    combined = Path("twitter_collection_results.json")
    combined.write_text(
        json.dumps(records, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    tweets = [
        tweet
        for record in records
        for tweet in record["response"]["tweets"]
    ]
    unique_ids = {tweet["id"] for tweet in tweets if tweet.get("id")}

    print("\n--- COLLECTION STATUS ---")
    print(f"Saved windows: {len(records)} / 46")
    print(f"Returned posts: {len(tweets)}")
    print(f"Unique post IDs: {len(unique_ids)}")
    print(f"File to upload: {combined.resolve()}")